# Engine Portability & Scale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/03_engine_scale.ipynb)

Write once, run anywhere. Same contract, different engines, identical results — plus dimensional modeling, incremental processing, parallel execution, backfill, and external logic hooks.

In [ ]:
import subprocess
import sys
import importlib
import urllib.request
import os

if importlib.util.find_spec("lakelogic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lakelogic[polars]"])
if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
from _setup import *

---
## 1. Engine-Agnostic Proof — Zero Code Changes

**The Problem:** You built your pipeline on Polars/DuckDB. Now it needs to run on Spark in production. Rewriting 2,000 lines of DataFrame logic isn't a weekend project.

**The Solution:** LakeLogic compiles SQL-first rules to each engine's dialect. Same contract, same results.

In [ ]:
contract = write_contract(
    """
version: 1.0.0
dataset: engine_test

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: email
      type: string
      required: true
    - name: score
      type: integer

quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: score_range
      sql: "score BETWEEN 0 AND 100"
""",
    "engine_test.yaml",
)

source_df = DataGenerator(contract).generate(rows=500, invalid_ratio=0.08)

# Run on Polars
p1 = DataProcessor(contract, engine="polars")
g1, b1 = p1.run(source_df)

# Run on DuckDB — same contract, zero changes
p2 = DataProcessor(contract, engine="duckdb")
g2, b2 = p2.run(source_df)

# Run on Spark — same contract, zero changes
p3 = DataProcessor(contract, engine="spark")
g3, b3 = p3.run(source_df)

In [ ]:
# The Proof
print("Engine Comparison")
print("=" * 40)
print(f"  Polars : good={len(g1)}, bad={len(b1)}")
print(f"  DuckDB : good={len(g2)}, bad={len(b2)}")
print(f"  Spark : good={len(g3)}, bad={len(b3)}")
print(f"  Match  : {len(g1) == len(g2) and len(b1) == len(b2) and len(g2) == len(g3) and len(b2) == len(b3)}")
print("\nSame contract. Same data. Same results. Zero code changes.")

---
## 2. Dimensional Modeling — SCD2, Merge, Overwrite

**The Problem:** Your dimension table needs history tracking. You manually build `MERGE INTO` SQL, manage `effective_from`/`effective_to` dates, and debug `is_current` flags by hand.

**The Solution:** Declare `materialization.strategy: scd2` in the contract — LakeLogic generates all SCD2 columns, applies merge logic, and manages version tracking. Zero SQL required.

In [ ]:
import yaml

# ── Show what a fully-declared dimensional contract looks like ────────
scd2_yaml = """
version: 1.0.0
dataset: dim_customers
info:
  title: gold_dim_customers
  target_layer: gold

primary_key: [customer_id]

model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
    - name: email
      type: string
    - name: tier
      type: string

materialization:
  strategy: scd2
  scd2:
    track_columns: [name, email, tier]
    timestamp_field: updated_at
    surrogate_key: _sk
    effective_from_field: effective_from
    effective_to_field: effective_to
    current_flag_field: is_current
    end_date_default: "9999-12-31"
    version_column: "_version"             # ROW_NUMBER per business key
    change_reason_column: "_change_reason" # "initial_load", "email,tier changes", etc.

    # Unknown member (late-arriving fact fallback)
    unknown_member:
      enabled: true
      surrogate_key_value: "-1"
"""

# Parse & validate the contract
from lakelogic.core.models import DataContract

scd2_contract = DataContract(**yaml.safe_load(scd2_yaml))

print("\u2500" * 60)
print("DIMENSIONAL MODELING: SCD2 Contract")
print("\u2500" * 60)
print(f"  Strategy       : {scd2_contract.materialization.strategy}")
print(f"  Primary key    : {scd2_contract.primary_key}")
print(f"  Track columns  : {scd2_contract.materialization.scd2['track_columns']}")
print(f"  Surrogate key  : {scd2_contract.materialization.scd2['surrogate_key']}")
print(f"  Effective from : {scd2_contract.materialization.scd2['effective_from_field']}")
print(f"  Effective to   : {scd2_contract.materialization.scd2['effective_to_field']}")
print(f"  Current flag   : {scd2_contract.materialization.scd2['current_flag_field']}")
print(f"  version        : {scd2_contract.materialization.scd2['version_column']}")
print(f"  change reason  : {scd2_contract.materialization.scd2['change_reason_column']}")
print(f"  unknown_member : {scd2_contract.materialization.scd2['unknown_member']}")


# ── Show all supported strategies ─────────────────────────────────────
strategies = {
    "append": "Fact tables — new rows added, never updated",
    "merge": "SCD Type 1 — upsert by natural key, latest value wins",
    "scd2": "SCD Type 2 — full history with effective dates",
    "overwrite": "Periodic snapshot — drop & replace on each run",
}

print(f"\n{'\u2500' * 60}")
print("ALL MATERIALIZATION STRATEGIES")
print("\u2500" * 60)
for strat, desc in strategies.items():
    marker = "\u2716" if strat == scd2_contract.materialization.strategy else " "
    print(f"  [{marker}] {strat:10s} — {desc}")
print("\n\u2705 All declared in YAML. No manual MERGE INTO SQL required.")

---
## 3. Incremental Processing — `pipeline_log` Watermark

**The Problem:** Your nightly job reprocesses 10 million rows even though only 500 changed. Compute costs scale with total volume instead of change volume.

**The Solution:** LakeLogic's `pipeline_log` watermark strategy tracks which files have been processed by their modification time. On the next run, only **new files** are loaded.

In [ ]:
import os
import shutil
import polars as pl

# ── Clean slate for demo ─────────────────────────────────────────────
DEMO_DIR = "./incremental_demo"
LANDING = f"{DEMO_DIR}/landing"

if os.path.exists(DEMO_DIR):
    shutil.rmtree(DEMO_DIR)
os.makedirs(LANDING, exist_ok=True)

# ── Contract with source.type = landing, load_mode = incremental ────
inc_contract = write_contract(
    """
version: 1.0.0
dataset: orders
info:
  title: bronze_orders
  target_layer: bronze

source:
  type: landing
  path: "./incremental_demo/landing"
  format: ndjson
  load_mode: incremental
  watermark_strategy: pipeline_log

metadata:
  run_log_dir: "./incremental_demo/logs"

model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
    - name: status
      type: string

quality:
  row_rules:
    - name: positive_amount
      sql: "amount > 0"
""",
    "orders_inc.yaml",
)

# ── FILE 1: 50 orders land in the landing zone ───────────────────────
batch1 = DataGenerator(inc_contract).generate(rows=50)
batch1.write_ndjson(f"{LANDING}/orders_batch_1.json")
print(f"\u2705 File 1: wrote {len(batch1)} rows to orders_batch_1.json")

# ── RUN 1: Initial load (no prior watermark) ────────────────────────
proc = DataProcessor(inc_contract, engine="polars")
g1, b1 = proc.run_source()
r1 = proc.last_report

print(f"\n{'=' * 50}")
print("RUN 1 (initial load)")
print(f"{'=' * 50}")
print(f"  Files in landing : {len(os.listdir(LANDING))}")
print(f"  Rows loaded      : {r1.get('counts', {}).get('source', '?')}")
print(f"  Good / Bad       : {r1.get('counts', {}).get('good', '?')} / {r1.get('counts', {}).get('quarantined', '?')}")

In [ ]:
import time

time.sleep(1)  # Ensure mtime of File 2 is strictly after Run 1's watermark

# ── FILE 2: 20 new orders arrive ────────────────────────────────────
batch2 = DataGenerator(inc_contract).generate(rows=20)
batch2.write_ndjson(f"{LANDING}/orders_batch_2.json")
print(f"\u2705 File 2: wrote {len(batch2)} rows to orders_batch_2.json")
print(f"  Landing zone now has: {os.listdir(LANDING)}")

# ── RUN 2: Only new files processed ─────────────────────────────────
g2, b2 = proc.run_source()
r2 = proc.last_report

print(f"\n{'=' * 50}")
print("RUN 2 (incremental)")
print(f"{'=' * 50}")
print(f"  Files in landing : {len(os.listdir(LANDING))} (70 total rows across 2 files)")
print("  Files processed  : 1 (only orders_batch_2.json \u2014 batch_1 already processed)")
print(f"  Rows loaded      : {r2.get('counts', {}).get('source', '?')}")
print(f"  Good / Bad       : {r2.get('counts', {}).get('good', '?')} / {r2.get('counts', {}).get('quarantined', '?')}")
print("\n\u2705 pipeline_log watermark: only new files are processed. No reprocessing.")

---
## 4. Parallel Processing — Concurrent Multi-Contract Execution

**The Problem:** You have 8 Bronze contracts with no dependencies between them. Running them sequentially takes 40 minutes.

**The Solution:** `pipeline.run(parallel=True)` groups contracts into dependency **waves** using topological sort. Contracts within the same wave execute concurrently via threads — layer ordering is preserved automatically.

In [ ]:
import os
import yaml
from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline import LakehousePipeline
from IPython.display import HTML, display

# ── Create inline contracts ──────────────────────────────────────────
DAG_DIR = "./parallel_demo"
os.makedirs(f"{DAG_DIR}/contracts/bronze", exist_ok=True)
os.makedirs(f"{DAG_DIR}/contracts/silver", exist_ok=True)

# Bronze: orders (independent)
write_contract(
    """
version: 1.0.0
dataset: orders
info:
  title: bronze_orders
  target_layer: bronze
source:
  type: landing
  path: "./parallel_demo/landing/orders"
  format: ndjson
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
""",
    f"{DAG_DIR}/contracts/bronze/orders.yaml",
)

# Bronze: customers (independent — runs in parallel with orders)
write_contract(
    """
version: 1.0.0
dataset: customers
info:
  title: bronze_customers
  target_layer: bronze
source:
  type: landing
  path: "./parallel_demo/landing/customers"
  format: ndjson
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
""",
    f"{DAG_DIR}/contracts/bronze/customers.yaml",
)

# Bronze: products (independent — runs in parallel with orders & customers)
write_contract(
    """
version: 1.0.0
dataset: products
info:
  title: bronze_products
  target_layer: bronze
source:
  type: landing
  path: "./parallel_demo/landing/products"
  format: ndjson
model:
  fields:
    - name: product_id
      type: integer
      required: true
    - name: name
      type: string
""",
    f"{DAG_DIR}/contracts/bronze/products.yaml",
)

# Silver: orders_enriched (depends on orders + customers → runs AFTER them)
write_contract(
    """
version: 1.0.0
dataset: orders_enriched
info:
  title: silver_orders_enriched
  target_layer: silver
source:
  type: table
  path: "./parallel_demo/lakehouse/bronze/bronze_orders"
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
downstream:
  - type: dashboard
    name: "Weekly Sales Performance"
    platform: power_bi
    url: "https://app.powerbi.com/..."
    owner: "marketing-analytics"
    
  - type: api
    name: "Order Tracking Service"
    platform: internal
    owner: "backend-team"

""",
    f"{DAG_DIR}/contracts/silver/orders_enriched.yaml",
)

# ── Create _system.yaml with dependency declarations ────────────────
system_yaml = {
    "domain": "demo",
    "system": "ecommerce",
    # ── External sources (for lineage visualization) ──────────────────────────
    "external_sources": [
        {
            "name": "Shopify API",
            "source_domain": "CRM Vendor",
            "catalog_path": "external_storage_path_or_api",
            "consumed_by": ["orders", "customers"],
        },
        {
            "name": "Products System Database",
            "source_domain": "Products Vendor",
            "catalog_path": "external_storage_path_or_api",
            "consumed_by": ["products"],
        },
    ],
    "contracts": [
        {"layer": "bronze", "entity": "orders", "path": "contracts/bronze/orders.yaml", "enabled": True},
        {"layer": "bronze", "entity": "customers", "path": "contracts/bronze/customers.yaml", "enabled": True},
        {"layer": "bronze", "entity": "products", "path": "contracts/bronze/products.yaml", "enabled": True},
        {
            "layer": "silver",
            "entity": "orders_enriched",
            "path": "contracts/silver/orders_enriched.yaml",
            "depends_on": ["orders", "customers"],
            "enabled": True,
        },
    ],
    "environments": {
        "local": {
            "catalog": "local",
            "storage_root": "./parallel_demo/lakehouse",
            "data_root": "./parallel_demo/lakehouse",
            "quarantine_root": "./parallel_demo/lakehouse/_quarantine",
        }
    },
    "storage": {"external_location_root": "./parallel_demo/lakehouse"},
}

sys_path = f"{DAG_DIR}/_system.yaml"
with open(sys_path, "w") as f:
    yaml.dump(system_yaml, f, default_flow_style=False)

# ── Build pipeline ──────────────────────────────────────────────────
registry = DomainRegistry.from_yaml(sys_path, environment="local", storage_mode="direct")
pipeline = LakehousePipeline(registry, engine="polars")

# ── Visualise the DAG — shows parallel waves ────────────────────────
display(HTML(pipeline.visualize_dag()))

# ── Show wave grouping ──────────────────────────────────────────────
from lakelogic.pipeline.runner import LakehousePipeline as _LP

bronze_contracts = [c for c in registry.contracts if c.layer == "bronze"]
waves = _LP._group_by_dependency_level(bronze_contracts)

print(f"\n{'\u2500' * 60}")
print("PARALLEL EXECUTION PLAN")
print("\u2500" * 60)
print(f"  Bronze layer: {len(bronze_contracts)} contracts")
for i, wave in enumerate(waves):
    entities = [c.entity for c in wave]
    print(f"  Wave {i}: [{', '.join(entities)}] \u2190 {'parallel' if len(entities) > 1 else 'sequential'}")

print("\n  Silver layer: orders_enriched")
print("  \u2514\u2500 depends_on: [orders, customers] \u2192 waits for Bronze to complete")

print("\n\u2705 pipeline.run(parallel=True) executes Wave 0 contracts concurrently.")
print("   Layer ordering (bronze \u2192 silver \u2192 gold) is always preserved.")

---
## 5. Backfill & Reprocessing — Targeted Late-Arriving Data

**The Problem:** A partner sent corrected data for last Tuesday. You need to reload just those records without blowing away the rest of the week.

**The Solution:** `run_source(reprocess_from=..., reprocess_to=...)` lets you surgically reload a date range or specific IDs — the incremental watermark is bypassed for that run only.

In [ ]:
import os
import shutil
from datetime import date, timedelta

# ── Clean slate ─────────────────────────────────────────────────────
BF_DIR = "./backfill_demo"
BF_LANDING = f"{BF_DIR}/landing"
if os.path.exists(BF_DIR):
    shutil.rmtree(BF_DIR)
os.makedirs(BF_LANDING, exist_ok=True)

# ── Contract with source.type = landing + reprocess column ─────────
backfill_contract = write_contract(
    """
version: 1.0.0
dataset: daily_events
info:
  title: bronze_daily_events
  target_layer: bronze

source:
  type: landing
  path: "./backfill_demo/landing/*.ndjson"
  format: ndjson

model:
  fields:
    - name: event_id
      type: integer
      required: true
    - name: event_date
      type: string
      required: true
    - name: payload
      type: string

materialization:
  reprocess_date_column: event_date

quality:
  row_rules:
    - name: has_payload
      sql: "payload IS NOT NULL"
""",
    "daily_events.yaml",
)

# ── Generate a week of events and write to landing ────────────────
today = date.today()
rows = []
for i in range(7):
    day = (today - timedelta(days=6 - i)).isoformat()
    for j in range(50):
        rows.append({"event_id": i * 50 + j, "event_date": day, "payload": f"data_{i}_{j}"})

full_week = pl.DataFrame(rows)
full_week.write_ndjson(f"{BF_LANDING}/events_full_week.ndjson")
print(f"Full dataset: {len(full_week)} rows across 7 days")
print(full_week.group_by("event_date").len().sort("event_date"))

# ── Full load first ──────────────────────────────────────────────────
bf_proc = DataProcessor(backfill_contract, engine="polars")
g_full, b_full = bf_proc.run_source()
r_full = bf_proc.last_report
print(f"\nFull load: {r_full.get('counts', {}).get('source', '?')} rows")

# ── Targeted backfill: reload just 2 days ──────────────────────────
target_start = (today - timedelta(days=3)).isoformat()
target_end = (today - timedelta(days=2)).isoformat()

g_bp, b_bp = bf_proc.run_source(
    reprocess_from=target_start,
    reprocess_to=target_end,
)
r_bp = bf_proc.last_report

print(f"\n{'=' * 50}")
print(f"BACKFILL: {target_start} to {target_end}")
print(f"{'=' * 50}")
print(f"  Rows reprocessed : {r_bp.get('counts', {}).get('source', '?')}")
print(f"  Good / Bad       : {r_bp.get('counts', {}).get('good', '?')} / {r_bp.get('counts', {}).get('bad', '?')}")
print("\n\u2705 Only the targeted date range was reprocessed \u2014 rest of the week untouched.")

---
## 6. External Logic — Custom Python Hooks

**The Problem:** Your Gold-layer transformation requires 200 lines of business logic — joins, pivots, ML scoring — that won't fit in a SQL rule.

**The Solution:** Declare `external_logic` in the contract. LakeLogic calls your custom Python function, feeds it the validated DataFrame, and then applies its own quality rules and lineage to the result.

In [ ]:
import yaml
from lakelogic.core.models import DataContract

# ── Show what an external_logic contract looks like ──────────────────
ext_yaml = """
version: 1.0.0
dataset: gold_revenue_summary
info:
  title: gold_revenue_summary
  target_layer: gold

source:
  type: table
  path: "lakehouse/silver/silver_orders_enriched"

model:
  fields:
    - name: region
      type: string
      required: true
    - name: total_revenue
      type: float
    - name: order_count
      type: integer

external_logic:
  type: python
  path: "transforms/revenue_summary.py"
  entrypoint: run
  args:
    fiscal_year: 2026
    include_refunds: false

quality:
  row_rules:
    - name: positive_revenue
      sql: "total_revenue >= 0"
"""

ext_contract = DataContract(**yaml.safe_load(ext_yaml))

print("\u2500" * 60)
print("EXTERNAL LOGIC: Contract-Driven Custom Transform")
print("\u2500" * 60)
print(f"  Hook type     : {ext_contract.external_logic.type}")
print(f"  Script path   : {ext_contract.external_logic.path}")
print(f"  Entrypoint    : {ext_contract.external_logic.entrypoint}()")
print(f"  Custom args   : {ext_contract.external_logic.args}")

print("\n  What happens at runtime:")
print("  1. LakeLogic loads data from source (silver_orders_enriched)")
print("  2. Calls transforms/revenue_summary.py → run(df, **args)")
print("  3. Your function returns a transformed DataFrame")
print("  4. LakeLogic validates it against the contract schema")
print("  5. Quality rules run (positive_revenue ≥ 0)")
print("  6. Good/Bad split + lineage injection + materialization")

# ── Show the Python script signature ───────────────────────────────
print(f"\n{'\u2500' * 60}")
print("transforms/revenue_summary.py (your custom code)")
print("\u2500" * 60)
print('''
def run(df, *, fiscal_year=2026, include_refunds=False, **kwargs):
    """Gold-layer aggregation — called by LakeLogic."""
    if not include_refunds:
        df = df.filter(pl.col("status") != "refunded")
    return df.group_by("region").agg(
        pl.col("amount").sum().alias("total_revenue"),
        pl.col("order_id").count().alias("order_count"),
    )
''')
print("\u2705 Your custom logic. LakeLogic's quality rules + lineage still apply.")

## What You Just Saw

| # | Feature | How |
|---|---------|-----|
| 1 | **Engine portability** | Same contract on Polars and DuckDB, identical results |
| 2 | **Dimensional modeling** | `strategy: scd2` — full history tracking declared in YAML |
| 3 | **Incremental processing** | `pipeline_log` watermark — only new files are loaded |
| 4 | **Parallel processing** | `pipeline.run(parallel=True)` — concurrent wave execution |
| 5 | **Backfill & reprocessing** | `reprocess_from`/`reprocess_to` — surgical date-range reload |
| 6 | **External logic** | `external_logic.type: python` — custom transforms with full lineage |

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.
